# SAM2-DEB-UNET: Dual Encoder + Boundary-Guided Refinement

Combined contribution của nhóm:
- **Dual Encoder**: SAM2-Hiera-L (1024×1024) + ConvNeXt-V2-Tiny (448×448)
- **BGHR Heads**: deep supervision S1/S2 + edge prediction + boundary refinement

Chạy trên **SAM2-UNet repo** (WZH0120) — giống các notebook BGHR trước, đã verify work.

**2 cách train:**
1. `--init_from best.pt` (bạn ấy) → chỉ cần fine-tune **20 epochs** (~1-2h)
2. Train từ đầu → **50 epochs** (~4-5h)


## 1. Clone SAM2-UNet repo + install

In [ ]:
!git clone https://github.com/WZH0120/SAM2-UNet.git
%cd SAM2-UNet
!pip install -r requirements.txt -q
!pip install -q timm albumentations tqdm gdown

Cloning into 'SAM2-UNet'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 316 (delta 85), reused 58 (delta 58), pack-reused 208 (from 2)
Receiving objects: 100% (316/316), 3.26 MiB | 29.51 MiB/s, done.
Resolving deltas: 100% (125/125), done.
/content/SAM2-UNet
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 57.9 MB/s eta 0:

## 2. Download SAM2 Hiera-L checkpoint

In [ ]:
!mkdir -p /content/checkpoints
!wget -q -O /content/checkpoints/sam2_hiera_large.pt \
  https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
!ls -lh /content/checkpoints

total 857M
-rw-r--r-- 1 root root 857M Jul 28  2024 sam2_hiera_large.pt


## 3. Dataset

In [ ]:
!mkdir -p /content/data
%cd /content/data
!gdown --fuzzy 'https://drive.google.com/file/d/1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb/view?usp=sharing' -O TrainDataset.zip
!gdown --fuzzy 'https://drive.google.com/file/d/1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao/view?usp=sharing' -O TestDataset.zip
!unzip -q -o TrainDataset.zip
!unzip -q -o TestDataset.zip
%cd /content/SAM2-UNet

/content/data
Downloading...
From (original): https://drive.google.com/uc?id=1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb
From (redirected): https://drive.google.com/uc?id=1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb&confirm=t&uuid=cfae37a6-4451-41c6-babe-1f7eb67f35cd
To: /content/data/TrainDataset.zip
100% 419M/419M [00:01<00:00, 221MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao
From (redirected): https://drive.google.com/uc?id=1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao&confirm=t&uuid=9ff4d677-9ac8-4eaf-9a46-0b87bd120c13
To: /content/data/TestDataset.zip
100% 343M/343M [00:02<00:00, 167MB/s]
/content/SAM2-UNet


In [ ]:
import os
from pathlib import Path
root = Path('/content/data')
for p in ['TrainDataset', 'TestDataset/Kvasir', 'TestDataset/CVC-ClinicDB',
          'TestDataset/CVC-ColonDB', 'TestDataset/CVC-300',
          'TestDataset/ETIS-LaribPolypDB']:
    full = root/p
    if full.exists():
        subs = sorted([x.name for x in full.iterdir() if x.is_dir()])
        print(f'{p}: {subs}')
    else:
        print(f'{p}: MISSING')

TrainDataset: ['image', 'masks']
TestDataset/Kvasir: ['images', 'masks']
TestDataset/CVC-ClinicDB: ['images', 'masks']
TestDataset/CVC-ColonDB: ['images', 'masks']
TestDataset/CVC-300: ['images', 'masks']
TestDataset/ETIS-LaribPolypDB: ['images', 'masks']


## 4. Ghi files vào repo

In [ ]:
%%writefile /content/SAM2-UNet/model_bg.py
"""
model_bg.py — SAM2-DEB-UNET: Dual encoder + Boundary-Guided refinement
========================================================================
Builds on model.py (dual-encoder design) by adding BGHR heads on top of
the decoder, WITHOUT modifying the encoder / fusion / decoder path.

Added components (all consume the final decoder feature `d` at 128 ch):
  • side2 head    — deep supervision from up3 output
  • side1 head    — deep supervision from up2 output
  • coarse_head   — coarse mask before BGHR refinement
  • BoundaryGuidedRefinement — edge prediction + edge attention + refine
  • final_head    — final refined logit (main output)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from timm.layers import trunc_normal_
from sam2.build_sam import build_sam2


class Adapter(nn.Module):
    def __init__(self, blk):
        super().__init__()
        self.block = blk
        dim = blk.attn.qkv.in_features
        self.prompt_learn = nn.Sequential(
            nn.Linear(dim, 32), nn.GELU(),
            nn.Linear(32, dim), nn.GELU(),
        )
        self._init_weights()

    def forward(self, x):
        return self.block(x + self.prompt_learn(x))

    def _init_weights(self):
        def _init(m):
            if isinstance(m, nn.Linear):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.bias, 0.0)
                nn.init.constant_(m.weight, 1.0)
        self.prompt_learn.apply(_init)


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        mid = mid_channels or out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)

    def forward(self, x1, x2=None):
        if x2 is not None:
            diffY = x1.size(2) - x2.size(2)
            diffX = x1.size(3) - x2.size(3)
            x2 = F.pad(x2, [diffX // 2, diffX - diffX // 2,
                             diffY // 2, diffY - diffY // 2])
            x1 = torch.cat([x1, x2], dim=1)
        return self.conv(self.up(x1))


class ConvBNReLU(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_planes, out_planes, kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_planes), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class BoundaryGuidedRefinement(nn.Module):
    """Predicts edge map → edge attention → refines feature for final mask."""
    def __init__(self, channels=128):
        super().__init__()
        self.edge_head = nn.Sequential(
            ConvBNReLU(channels, channels, kernel_size=3, padding=1),
            nn.Conv2d(channels, 1, kernel_size=1),
        )
        self.edge_attention = nn.Sequential(
            nn.Conv2d(1, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.Sigmoid(),
        )
        self.refine = nn.Sequential(
            ConvBNReLU(channels + 1, channels, kernel_size=3, padding=1),
            ConvBNReLU(channels, channels, kernel_size=3, padding=1),
        )

    def forward(self, feat):
        edge_logit = self.edge_head(feat)
        edge_prob = torch.sigmoid(edge_logit)
        att = self.edge_attention(edge_prob)
        feat = feat + feat * att
        refined = self.refine(torch.cat([feat, edge_prob], dim=1))
        return refined, edge_logit


class SAM2UNeXT_BG(nn.Module):
    """
    SAM2-UNeXT + BGHR. Encoder/fusion/decoder identical to original SAM2UNeXT;
    BGHR adds deep supervision + edge prediction + refinement.

    Forward signature: model(x_sam, x_cnx)  -> dict.
    Returns dict with: final, coarse, s1, s2, edge — all (B, 1, 352, 352).
    """
    _SAM2_CHANNELS = (144, 288, 576, 1152)
    _CONVNEXT_FINAL_CH = 768

    def __init__(self,
                 sam2_checkpoint=None,
                 sam2_cfg="sam2_hiera_l.yaml",
                 convnext_pretrained=True):
        super().__init__()

        # ---------- SAM2 Hiera-L (frozen + adapters) ----------
        model = build_sam2(sam2_cfg, sam2_checkpoint)
        for attr in ("sam_mask_decoder", "sam_prompt_encoder", "memory_encoder",
                     "memory_attention", "mask_downsample", "obj_ptr_tpos_proj",
                     "obj_ptr_proj"):
            if hasattr(model, attr):
                delattr(model, attr)
        if hasattr(model.image_encoder, "neck"):
            del model.image_encoder.neck

        self.sam = model.image_encoder.trunk
        for p in self.sam.parameters():
            p.requires_grad = False
        self.sam.blocks = nn.Sequential(*[Adapter(blk) for blk in self.sam.blocks])

        # ---------- ConvNeXt-V2-Tiny (partial freeze: stage 2/3 unfrozen) ----------
        self.convnext = timm.create_model(
            "convnextv2_tiny.fcmae_ft_in22k_in1k",
            pretrained=convnext_pretrained, features_only=True,
        )
        self._freeze_convnext_partial()

        # ---------- Dense Glue (project ConvNeXt 768 → SAM2 channel widths) ----------
        in_ch = self._CONVNEXT_FINAL_CH
        self.align1 = nn.Conv2d(in_ch, self._SAM2_CHANNELS[0], 1)
        self.align2 = nn.Conv2d(in_ch, self._SAM2_CHANNELS[1], 1)
        self.align3 = nn.Conv2d(in_ch, self._SAM2_CHANNELS[2], 1)
        self.align4 = nn.Conv2d(in_ch, self._SAM2_CHANNELS[3], 1)

        # ---------- Reduce to 128 ch ----------
        self.reduce1 = nn.Conv2d(self._SAM2_CHANNELS[0] * 2, 128, 1)
        self.reduce2 = nn.Conv2d(self._SAM2_CHANNELS[1] * 2, 128, 1)
        self.reduce3 = nn.Conv2d(self._SAM2_CHANNELS[2] * 2, 128, 1)
        self.reduce4 = nn.Conv2d(self._SAM2_CHANNELS[3] * 2, 128, 1)

        # ---------- UNet decoder ----------
        self.up1 = Up(256, 128)
        self.up2 = Up(256, 128)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 128)

        # ---------- BGHR heads (the new contributions) ----------
        self.side2       = nn.Conv2d(128, 1, kernel_size=1)
        self.side1       = nn.Conv2d(128, 1, kernel_size=1)
        self.coarse_head = nn.Conv2d(128, 1, kernel_size=1)
        self.boundary_refine = BoundaryGuidedRefinement(channels=128)
        self.final_head  = nn.Conv2d(128, 1, kernel_size=1)

    def _freeze_convnext_partial(self):
        for name, p in self.convnext.named_parameters():
            if name.startswith("stem") or name.startswith("stages.0") or name.startswith("stages.1"):
                p.requires_grad = False
            else:
                p.requires_grad = True
        total  = sum(p.numel() for p in self.convnext.parameters())
        frozen = sum(p.numel() for p in self.convnext.parameters() if not p.requires_grad)
        print(f"[ConvNeXt-V2] {frozen:,} / {total:,} params frozen "
              f"({100*frozen/total:.1f}%)")

    def forward(self, x_sam, x_cnx):
        # Encoders
        x1_s, x2_s, x3_s, x4_s = self.sam(x_sam)
        x_c = self.convnext(x_cnx)[-1]

        # Dense glue
        x1_c = F.interpolate(self.align1(x_c), size=x1_s.shape[-2:], mode="bilinear", align_corners=False)
        x2_c = F.interpolate(self.align2(x_c), size=x2_s.shape[-2:], mode="bilinear", align_corners=False)
        x3_c = F.interpolate(self.align3(x_c), size=x3_s.shape[-2:], mode="bilinear", align_corners=False)
        x4_c = F.interpolate(self.align4(x_c), size=x4_s.shape[-2:], mode="bilinear", align_corners=False)

        # Fuse
        x1 = self.reduce1(torch.cat([x1_s, x1_c], dim=1))
        x2 = self.reduce2(torch.cat([x2_s, x2_c], dim=1))
        x3 = self.reduce3(torch.cat([x3_s, x3_c], dim=1))
        x4 = self.reduce4(torch.cat([x4_s, x4_c], dim=1))

        # Decoder + deep supervision
        d = self.up4(x4)
        d = self.up3(d, x3)
        s2 = self.side2(d)
        d = self.up2(d, x2)
        s1 = self.side1(d)
        d = self.up1(d, x1)

        # BGHR
        coarse = self.coarse_head(d)
        refined, edge = self.boundary_refine(d)
        final = self.final_head(refined)

        # Upsample all to 352x352
        T = (352, 352)
        return {
            "final":  F.interpolate(final,  size=T, mode="bilinear", align_corners=False),
            "coarse": F.interpolate(coarse, size=T, mode="bilinear", align_corners=False),
            "s1":     F.interpolate(s1,     size=T, mode="bilinear", align_corners=False),
            "s2":     F.interpolate(s2,     size=T, mode="bilinear", align_corners=False),
            "edge":   F.interpolate(edge,   size=T, mode="bilinear", align_corners=False),
        }


def load_friend_checkpoint(model, ckpt_path, verbose=True):
    """
    Load friend's SAM2UNeXT checkpoint into SAM2UNeXT_BG.
    Matches keys with same name AND same shape; skips BGHR-specific keys
    (side1, side2, coarse_head, boundary_refine, final_head) plus friend's
    'head' layer (which we don't use).
    """
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd_src = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    sd_dst = model.state_dict()
    loaded, mismatched, missing = 0, 0, 0
    for k, v in sd_src.items():
        if k in sd_dst and sd_dst[k].shape == v.shape:
            sd_dst[k] = v
            loaded += 1
        elif k in sd_dst:
            mismatched += 1
        else:
            missing += 1
    model.load_state_dict(sd_dst)
    if verbose:
        new_keys = sum(1 for k in sd_dst if k not in sd_src)
        print(f"[load_friend_checkpoint] loaded={loaded}  "
              f"src_extra={missing} (e.g. friend's 'head')  "
              f"dst_new={new_keys} (BGHR heads, random init)")
    return model


if __name__ == "__main__":
    import sys
    print("SAM2-UNeXT-BG smoke test")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    try:
        model = SAM2UNeXT_BG(sam2_checkpoint=None, convnext_pretrained=True).to(device)
    except Exception as e:
        print(f"[ERROR] {e}"); sys.exit(1)
    model.eval()
    x_sam = torch.randn(1, 3, 1024, 1024, device=device)
    x_cnx = torch.randn(1, 3,  448,  448, device=device)
    with torch.no_grad():
        out = model(x_sam, x_cnx)
    for k, v in out.items():
        print(f"  {k:10s}: {tuple(v.shape)}")
    total = sum(p.numel() for p in model.parameters()) / 1e6
    train = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"Total: {total:.2f}M | Trainable: {train:.2f}M")


Writing /content/SAM2-UNet/model_bg.py


In [ ]:
%%writefile /content/SAM2-UNet/dataset_bg.py
"""
dataset.py — Polyp Segmentation Dataset (PraNet Protocol Edition)
==================================================================
Returns three tensors per sample:
  • img_sam   : (3, 1024, 1024)  — normalised RGB for SAM2
  • img_cnx   : (3,  448,  448)  — normalised RGB for ConvNeXt-V2
  • mask      : (1,  352,  352)  — binary ground-truth mask {0, 1}

Auto-detects image and mask extensions to support PraNet's multi-dataset format.
"""

import os
from pathlib import Path
from typing import Literal

import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
from torch.utils.data import Dataset, DataLoader


# ---------------------------------------------------------------------------
# Image-net mean / std used for both encoder inputs
# ---------------------------------------------------------------------------
_IMAGENET_MEAN = (0.485, 0.456, 0.406)
_IMAGENET_STD  = (0.229, 0.224, 0.225)


# ---------------------------------------------------------------------------
# Augmentation pipelines
# ---------------------------------------------------------------------------

def _train_augmentations() -> A.Compose:
    """
    Standard augmentation suite for medical polyp images.
    Đã fix các cảnh báo của Albumentations phiên bản mới nhất.
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            scale=(0.9, 1.1),
            translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
            rotate=(-15, 15),
            p=0.4, # Đã xóa 'mode' gây cảnh báo
        ),
        A.ElasticTransform(alpha=80, sigma=10, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.2),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.GaussNoise(p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
        A.CoarseDropout(
            num_holes_range=(1, 4),
            hole_height_range=(1, 32),
            hole_width_range=(1, 32),
            p=0.1, # Đã xóa 'fill_value' gây cảnh báo
        ),
    ], additional_targets={"mask": "mask"})


def _val_augmentations() -> A.Compose:
    """No spatial or photometric augmentation at validation time."""
    return A.Compose([], additional_targets={"mask": "mask"})


def _to_tensor_normalise(size: int) -> A.Compose:
    """
    Resize to *size* × *size*, convert to float32 tensor in [0, 1],
    then apply ImageNet normalisation.
    """
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
        ToTensorV2(),
    ])


def _mask_resize(size: int) -> A.Compose:
    """Resize a binary mask using nearest-neighbour interpolation."""
    return A.Compose([
        A.Resize(size, size, interpolation=cv2.INTER_NEAREST),
    ])


# ---------------------------------------------------------------------------
# Dataset class
# ---------------------------------------------------------------------------

class PolypDataset(Dataset):
    """
    Dataset class robust to varying image extensions (.jpg, .png, .tif).
    Matches images to masks based purely on the filename stem.
    """

    def __init__(
        self,
        root: str | Path,
        mode: Literal["train", "val", "test"] = "train",
        img_size_sam: int = 1024,
        img_size_cnx: int = 448,
        mask_size: int = 352,
    ) -> None:
        super().__init__()
        self.root = Path(root)
        self.mode = mode

        # --- LOGIC TỰ DÒ THƯ MỤC ẢNH ---
        if (self.root / "images").exists():
            self.img_dir = self.root / "images"
        elif (self.root / "image").exists():
            self.img_dir = self.root / "image"
        else:
            raise RuntimeError(f"Không tìm thấy thư mục 'images' hoặc 'image' trong {self.root}")

        # --- LOGIC TỰ DÒ THƯ MỤC MASK ---
        if (self.root / "masks").exists():
            self.mask_dir = self.root / "masks"
        elif (self.root / "mask").exists():
            self.mask_dir = self.root / "mask"
        else:
            raise RuntimeError(f"Không tìm thấy thư mục 'masks' hoặc 'mask' trong {self.root}")

        self.img_paths = sorted([p for p in self.img_dir.iterdir() if p.is_file()])
        self.stems = [p.stem for p in self.img_paths]

        if len(self.img_paths) == 0:
            raise RuntimeError(f"Không tìm thấy ảnh nào trong {self.img_dir}")

        self.aug = _train_augmentations() if mode == "train" else _val_augmentations()
        self.to_sam = _to_tensor_normalise(img_size_sam)
        self.to_cnx = _to_tensor_normalise(img_size_cnx)
        self.resize_mask = _mask_resize(mask_size)

    def __len__(self) -> int:
        return len(self.img_paths)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        img_path = self.img_paths[idx]
        stem = img_path.stem

        # Quét thư mục masks để tìm file có chung tên (stem) bất chấp đuôi
        mask_candidates = list(self.mask_dir.glob(f"{stem}.*"))
        if not mask_candidates:
            raise FileNotFoundError(f"Không tìm thấy mask tương ứng cho ảnh: {stem}")
        mask_path = mask_candidates[0]

        # --- Load image (BGR → RGB) and mask ---
        img  = cv2.cvtColor(cv2.imread(str(img_path),  cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise RuntimeError(f"Không thể đọc file ảnh: {img_path}")
        if mask is None:
            raise RuntimeError(f"Không thể đọc file mask: {mask_path}")

        # Binarise mask
        mask = (mask > 127).astype(np.uint8)

        # --- Augmentation ---
        augmented = self.aug(image=img, mask=mask)
        aug_img   = augmented["image"]
        aug_mask  = augmented["mask"]

        # --- Tensors generation ---
        img_sam: torch.Tensor = self.to_sam(image=aug_img)["image"]
        img_cnx: torch.Tensor = self.to_cnx(image=aug_img)["image"]

        resized_mask = self.resize_mask(image=aug_mask)["image"]
        gt_mask: torch.Tensor = torch.from_numpy(resized_mask.astype(np.float32)).unsqueeze(0)

        return {
            "img_sam" : img_sam,    # (3, 1024, 1024)
            "img_cnx" : img_cnx,    # (3,  448,  448)
            "mask"    : gt_mask,    # (1,  352,  352)
            "stem"    : stem,       # string
        }


# ---------------------------------------------------------------------------
# Convenience DataLoader factory
# ---------------------------------------------------------------------------

def get_loader(
    root: str | Path,
    batch_size: int = 4,
    mode: Literal["train", "val", "test"] = "train",
    num_workers: int = 4,
    pin_memory: bool = True,
    **dataset_kwargs,
) -> DataLoader:
    dataset = PolypDataset(root, mode=mode, **dataset_kwargs)
    shuffle = (mode == "train")

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=(mode == "train"),
    )
    print(
        f"[DataLoader] mode={mode} | samples={len(dataset)} | "
        f"batches={len(loader)} | batch_size={batch_size}"
    )
    return loader


# ---------------------------------------------------------------------------
# Quick sanity check
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    import sys

    root = sys.argv[1] if len(sys.argv) > 1 else "data/TrainDataset"

    try:
        loader = get_loader(root, batch_size=2, mode="train", num_workers=0)
        batch = next(iter(loader))
        print("img_sam :", batch["img_sam"].shape,  batch["img_sam"].dtype)
        print("img_cnx :", batch["img_cnx"].shape,  batch["img_cnx"].dtype)
        print("mask    :", batch["mask"].shape,    batch["mask"].dtype)
        print("stems   :", batch["stem"])
        print("✓ Dataset hoạt động hoàn hảo!")
    except Exception as e:
        print(f"Lỗi khởi tạo dataset: {e}")

Writing /content/SAM2-UNet/dataset_bg.py


In [ ]:
%%writefile /content/SAM2-UNet/train_bg.py
"""
train_bg.py — Train SAM2-UNeXT-BG (dual encoder + BGHR heads)
==============================================================
Extends friend's train.py with:
  • Multi-head loss: final (1.0) + coarse (0.5) + s1 (0.3) + s2 (0.3)
                   + edge (0.3) + boundary (0.3)
  • Optional --init_from to start from friend's checkpoint (matching keys)
  • Otherwise identical training protocol (AdamW + Cosine + AMP + BCE+Dice)
"""

import argparse, time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

try:
    from tqdm import tqdm
except ImportError:
    class tqdm:
        def __init__(self, iterable=None, **kw): self._it = iterable
        def __iter__(self): return iter(self._it)
        def set_postfix(self, **kw): pass
        def __len__(self): return len(self._it)

from model_bg import SAM2UNeXT_BG, load_friend_checkpoint
from dataset_bg import get_loader


# ---------- Loss components ----------
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(logits.size(0), -1)
        t = targets.view(targets.size(0), -1)
        inter = (p * t).sum(dim=1)
        dice = (2*inter + self.smooth) / (p.sum(dim=1) + t.sum(dim=1) + self.smooth)
        return 1.0 - dice.mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, w_bce=0.5, w_dice=0.5):
        super().__init__()
        self.w_bce, self.w_dice = w_bce, w_dice
        self.bce, self.dice = nn.BCEWithLogitsLoss(), DiceLoss()
    def forward(self, logits, targets):
        return self.w_bce * self.bce(logits, targets) + self.w_dice * self.dice(logits, targets)

def mask_to_boundary(mask, k=5):
    mask = (mask > 0.5).float()
    pad = k // 2
    dilated = F.max_pool2d(mask, k, stride=1, padding=pad)
    eroded  = -F.max_pool2d(-mask, k, stride=1, padding=pad)
    return (dilated - eroded).clamp(0, 1)

def edge_loss(pred_edge_logit, gt_mask):
    gt_edge = mask_to_boundary(gt_mask)
    p = torch.sigmoid(pred_edge_logit)
    inter = (p * gt_edge).sum(dim=(2,3))
    denom = p.sum(dim=(2,3)) + gt_edge.sum(dim=(2,3))
    return (1 - (2*inter + 1) / (denom + 1)).mean()

def boundary_loss(pred_mask_logit, gt_mask):
    gt_b = mask_to_boundary(gt_mask)
    pb = torch.sigmoid(pred_mask_logit)
    pb_b = mask_to_boundary(pb)
    inter = (pb_b * gt_b).sum(dim=(2,3))
    denom = pb_b.sum(dim=(2,3)) + gt_b.sum(dim=(2,3))
    return (1 - (2*inter + 1) / (denom + 1)).mean()


_BCE_DICE = BCEDiceLoss(0.5, 0.5)

def compute_loss(outputs, mask, w):
    l_final  = _BCE_DICE(outputs["final"],  mask)
    l_coarse = _BCE_DICE(outputs["coarse"], mask)
    l_s1     = _BCE_DICE(outputs["s1"],     mask)
    l_s2     = _BCE_DICE(outputs["s2"],     mask)
    l_edge   = edge_loss(outputs["edge"], mask)
    l_bd     = boundary_loss(outputs["final"], mask)
    total = (w["final"]    * l_final
           + w["coarse"]   * l_coarse
           + w["s1"]       * l_s1
           + w["s2"]       * l_s2
           + w["edge"]     * l_edge
           + w["boundary"] * l_bd)
    return total


@torch.no_grad()
def dice_score(logits, targets, thr=0.5, smooth=1.0):
    p = (torch.sigmoid(logits) > thr).float().view(logits.size(0), -1)
    t = targets.view(targets.size(0), -1)
    inter = (p * t).sum(dim=1)
    return ((2*inter + smooth) / (p.sum(dim=1) + t.sum(dim=1) + smooth)).mean().item()


def train_one_epoch(model, loader, optimizer, weights, scaler, device, epoch):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch:03d} [train]", leave=False)
    for batch in pbar:
        x_sam = batch["img_sam"].to(device, non_blocking=True)
        x_cnx = batch["img_cnx"].to(device, non_blocking=True)
        mask  = batch["mask"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            outputs = model(x_sam, x_cnx)
            loss = compute_loss(outputs, mask, weights)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, weights, device, epoch):
    model.eval()
    total_loss, total_dice = 0.0, 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch:03d} [ val ]", leave=False)
    for i, batch in enumerate(pbar):
        x_sam = batch["img_sam"].to(device, non_blocking=True)
        x_cnx = batch["img_cnx"].to(device, non_blocking=True)
        mask  = batch["mask"].to(device, non_blocking=True)
        with autocast():
            outputs = model(x_sam, x_cnx)
            loss = compute_loss(outputs, mask, weights)
        total_loss += loss.item()
        total_dice += dice_score(outputs["final"], mask)
        pbar.set_postfix(dice=f"{total_dice/(i+1):.4f}")
    n = len(loader)
    return total_loss / n, total_dice / n


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    train_loader = get_loader(args.train_dir, batch_size=args.batch_size,
                              mode="train", num_workers=args.num_workers)
    val_loader   = get_loader(args.val_dir, batch_size=1,
                              mode="val", num_workers=args.num_workers)

    model = SAM2UNeXT_BG(sam2_checkpoint=args.sam2_ckpt,
                         convnext_pretrained=True).to(device)

    if args.init_from and Path(args.init_from).exists():
        load_friend_checkpoint(model, args.init_from)

    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f"Trainable tensors: {len(trainable)}")
    print(f"Trainable params : {sum(p.numel() for p in trainable)/1e6:.2f}M")

    optimizer = AdamW(trainable, lr=args.lr, weight_decay=args.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=args.lr * 1e-2)
    scaler = GradScaler()

    weights = {"final": 1.0, "coarse": 0.5, "s1": 0.3, "s2": 0.3,
               "edge": 0.3, "boundary": 0.3}
    print(f"Loss weights: {weights}")

    save_dir = Path(args.save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    start_epoch, best_dice = 1, 0.0

    if args.resume and Path(args.resume).exists():
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimiser"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        start_epoch = ckpt["epoch"] + 1
        best_dice = ckpt["best_dice"]
        print(f"Resumed from {args.resume}, epoch {start_epoch}, best_dice={best_dice:.4f}")

    print("\n" + "=" * 60 + "\nStarting training\n" + "=" * 60)
    for epoch in range(start_epoch, args.epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, weights, scaler, device, epoch)
        val_loss, val_dice = validate(model, val_loader, weights, device, epoch)
        scheduler.step()

        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch:03d}/{args.epochs:03d}  "
              f"| train={train_loss:.4f} | val={val_loss:.4f} | dice={val_dice:.4f} "
              f"| lr={lr_now:.2e} | {elapsed:.0f}s")
        if device.type == "cuda":
            used = torch.cuda.memory_allocated() / 1e9
            print(f"          VRAM: {used:.1f} GB")

        payload = {"epoch": epoch, "model": model.state_dict(),
                   "optimiser": optimizer.state_dict(),
                   "scheduler": scheduler.state_dict(),
                   "scaler": scaler.state_dict(),
                   "best_dice": best_dice, "val_dice": val_dice}
        torch.save(payload, save_dir / "last.pt")
        if val_dice > best_dice:
            best_dice = val_dice
            payload["best_dice"] = best_dice
            torch.save(payload, save_dir / "best.pt")
            print(f"          ★ New best Dice: {best_dice:.4f}")

    print(f"\nDone. Best val Dice = {best_dice:.4f}")


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--train_dir",   type=str, required=True)
    p.add_argument("--val_dir",     type=str, required=True)
    p.add_argument("--sam2_ckpt",   type=str, default=None)
    p.add_argument("--init_from",   type=str, default=None,
                   help="Path to friend's checkpoint to init shared weights")
    p.add_argument("--save_dir",    type=str, default="checkpoints_bg")
    p.add_argument("--resume",      type=str, default=None)
    p.add_argument("--epochs",       type=int,   default=50)
    p.add_argument("--batch_size",   type=int,   default=4)
    p.add_argument("--lr",           type=float, default=1e-4)
    p.add_argument("--weight_decay", type=float, default=1e-4)
    p.add_argument("--num_workers",  type=int,   default=4)
    return p.parse_args()


if __name__ == "__main__":
    train(parse_args())



Overwriting /content/SAM2-UNet/train_bg.py


In [ ]:
%%writefile /content/SAM2-UNet/test_bg.py
"""
test_bg.py — Evaluate SAM2-UNeXT-BG on all 5 polyp test datasets
"""
import os, argparse
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from model_bg import SAM2UNeXT_BG

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def make_tf(size):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

TF_SAM = make_tf(1024)
TF_CNX = make_tf(448)


def cal_metrics(pred, gt):
    pred_b = (pred >= 0.5).astype(np.float32)
    gt_b   = (gt   >= 0.5).astype(np.float32)
    inter  = (pred_b * gt_b).sum()
    dice   = (2*inter + 1e-6) / (pred_b.sum() + gt_b.sum() + 1e-6)
    iou    = (inter + 1e-6) / (pred_b.sum() + gt_b.sum() - inter + 1e-6)
    mae    = np.abs(pred - gt).mean()
    return dice, iou, mae


def find_mask(mask_dir, stem):
    for ext in [".png", ".jpg", ".bmp", ".tif"]:
        p = os.path.join(mask_dir, stem + ext)
        if os.path.exists(p):
            return p
    return None


def find_dir(parent, names):
    for n in names:
        p = os.path.join(parent, n)
        if os.path.isdir(p):
            return p
    return None


def evaluate(model, img_dir, mask_dir, device):
    img_files = sorted(os.listdir(img_dir))
    dices, ious, maes = [], [], []
    with torch.no_grad():
        for fname in img_files:
            stem = os.path.splitext(fname)[0]
            mask_path = find_mask(mask_dir, stem)
            if mask_path is None:
                continue
            img = Image.open(os.path.join(img_dir, fname)).convert("RGB")
            gt  = np.array(Image.open(mask_path).convert("L")) / 255.0
            orig_hw = gt.shape

            x_sam = TF_SAM(img).unsqueeze(0).to(device)
            x_cnx = TF_CNX(img).unsqueeze(0).to(device)

            out = model(x_sam, x_cnx)
            pred = torch.sigmoid(out["final"])
            pred = F.interpolate(pred, size=orig_hw, mode="bilinear", align_corners=False)
            pred = pred.squeeze().cpu().numpy()

            d, i, m = cal_metrics(pred, gt)
            dices.append(d); ious.append(i); maes.append(m)
    return np.mean(dices), np.mean(ious), np.mean(maes)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--checkpoint", type=str, required=True)
    p.add_argument("--test_root",  type=str, required=True,
                   help="Parent dir containing dataset subdirs")
    p.add_argument("--sam2_cfg",   type=str, default="configs/sam2.1/sam2.1_hiera_l.yaml")
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SAM2UNeXT_BG(sam2_checkpoint=None, sam2_cfg=args.sam2_cfg,
                         convnext_pretrained=False).to(device)
    ckpt = torch.load(args.checkpoint, map_location=device)
    sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    model.load_state_dict(sd)
    model.eval()

    datasets = ["Kvasir", "CVC-ClinicDB", "CVC-ColonDB", "CVC-300", "ETIS-LaribPolypDB"]
    for ds in datasets:
        ds_root = os.path.join(args.test_root, ds)
        if not os.path.isdir(ds_root):
            print(f"Skipping {ds} (not found)"); continue
        img_dir  = find_dir(ds_root, ["images", "image"])
        mask_dir = find_dir(ds_root, ["masks", "mask"])
        if not (img_dir and mask_dir):
            print(f"Skipping {ds} (missing img/mask)"); continue
        dice, iou, mae = evaluate(model, img_dir, mask_dir, device)
        print(f"========================= {ds} =========================")
        print(f"mDice : {dice:.3f}")
        print(f"mIoU  : {iou:.3f}")
        print(f"MAE   : {mae:.3f}")


Overwriting /content/SAM2-UNet/test_bg.py


In [ ]:
%cd /content/SAM2-UNet
!ls model_bg.py dataset_bg.py train_bg.py test_bg.py

/content/SAM2-UNet
dataset_bg.py  model_bg.py  test_bg.py	train_bg.py


## 5. Param count check

In [ ]:
%cd /content/SAM2-UNet
import sys; sys.path.insert(0, '.')
from model_bg import SAM2UNeXT_BG
m = SAM2UNeXT_BG(sam2_checkpoint='/content/checkpoints/sam2_hiera_large.pt',
                 convnext_pretrained=True)
total = sum(p.numel() for p in m.parameters()) / 1e6
train = sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6
print(f'Total params    : {total:.2f}M')
print(f'Trainable params: {train:.2f}M')
del m

/content/SAM2-UNet


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

[ConvNeXt-V2] 4,896 / 27,864,960 params frozen (0.0%)
Total params    : 245.87M
Trainable params: 33.71M


## 6. Smoke test — 1 epoch, batch 2

In [ ]:
%cd /content/SAM2-UNet

# Fix import trong train_bg.py: dataset -> dataset_bg
import re
with open('train_bg.py') as f: s = f.read()
s = s.replace('from dataset  import get_loader', 'from dataset_bg import get_loader')
with open('train_bg.py', 'w') as f: f.write(s)

!python train_bg.py \
  --train_dir /content/data/TrainDataset \
  --val_dir   /content/data/TestDataset/Kvasir \
  --sam2_ckpt /content/checkpoints/sam2_hiera_large.pt \
  --save_dir  /content/ckpt/smoke \
  --epochs 1 --batch_size 2

/content/SAM2-UNet
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
[DataLoader] mode=train | samples=1450 | batches=725 | batch_size=2
[DataLoader] mode=val | samples=100 | batches=100 | batch_size=1
[ConvNeXt-V2] 4,896 / 27,864,960 params frozen (0.0%)
Trainable tensors: 446
Trainable params : 33.71M
Loss weights: {'final': 1.0, 'coarse': 0.5, 's1': 0.3, 's2': 0.3, 'edge': 0.3, 'boundary': 0.3}

Starting training
Epoch 001/001  | train=1.1269 | val=0.7421 | dice=0.8893 | lr=1.00e-06 | 190s
          VRAM: 1.4 GB
          ★ New best Dice: 0.8893

Done. Best val Dice = 0.8893


## Train từ đầu — 50 epochs

In [ ]:
%cd /content/SAM2-UNet
!python train_bg.py \
  --train_dir /content/data/TrainDataset \
  --val_dir   /content/data/TestDataset/Kvasir \
  --sam2_ckpt /content/checkpoints/sam2_hiera_large.pt \
  --save_dir  /content/ckpt/bg_scratch \
  --epochs 50 --batch_size 8 --lr 1e-4

/content/SAM2-UNet
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
[DataLoader] mode=train | samples=1450 | batches=181 | batch_size=8
[DataLoader] mode=val | samples=100 | batches=100 | batch_size=1
[ConvNeXt-V2] 4,896 / 27,864,960 params frozen (0.0%)
Trainable tensors: 446
Trainable params : 33.71M
Loss weights: {'final': 1.0, 'coarse': 0.5, 's1': 0.3, 's2': 0.3, 'edge': 0.3, 'boundary': 0.3}

Starting training
Epoch 001/050  | train=1.3205 | val=0.9353 | dice=0.8832 | lr=9.99e-05 | 161s
          VRAM: 1.4 GB
          ★ New best Dice: 0.8832
Epoch 002/050  | train=0.9354 | val=0.7120 | dice=0.8962 | lr=9.96e-05 | 159s
          VRAM: 1.4 GB
          ★ New best Dice: 0.8962
Epoch 003/050  | train=0.7764 | val=0.6010 | dice=0.9090 | lr=9.91e-05 | 159s
          VRAM: 1.4 GB
          ★ New best Dice: 0.9090
Epoch 004/050  | train=0.6515 | val=0.5301 | dice=0.9091 | lr=9.84e-05 | 159s
          VRAM: 1.4 GB
          ★ New best Dice: 0.9091
Epoch 005/050  | train=0.5834 | val=0.5559 | dice=

In [ ]:
with open('/content/SAM2-UNet/test_bg.py') as f:
    s = f.read()
s = s.replace('default="configs/sam2.1/sam2.1_hiera_l.yaml"', 'default="sam2_hiera_l.yaml"')
with open('/content/SAM2-UNet/test_bg.py', 'w') as f:
    f.write(s)
print("Fixed.")


Fixed.


## 8. Eval trên 5 datasets

In [ ]:
%cd /content/SAM2-UNet
CKPT='/content/ckpt/bg_scratch/best.pt'  # đổi sang bg_scratch nếu dùng 7b

!python test_bg.py \
  --checkpoint $CKPT \
  --test_root  /content/data/TestDataset

/content/SAM2-UNet
[ConvNeXt-V2] 4,896 / 27,864,960 params frozen (0.0%)
========================= Kvasir =========================
mDice : 0.935
mIoU  : 0.891
MAE   : 0.022
========================= CVC-ClinicDB =========================
mDice : 0.916
mIoU  : 0.868
MAE   : 0.009
========================= CVC-ColonDB =========================
mDice : 0.811
mIoU  : 0.733
MAE   : 0.032
========================= CVC-300 =========================
mDice : 0.876
mIoU  : 0.814
MAE   : 0.010
========================= ETIS-LaribPolypDB =========================
mDice : 0.828
mIoU  : 0.756
MAE   : 0.013
